In [0]:
df = spark.read.parquet(
    "/Volumes/workspace/default/silver/ecomm-behavior"
)

df.printSchema()
df.show(2)

In [0]:
df.select("event_type").distinct().show()


In [0]:
df.count() # about 100 million reocrds

In [0]:
df.groupBy("event_type").count().show()

In [0]:
df.groupBy("event_type").count().show()

In [0]:
%sql
-- this code will not work, need adls or s3 or gcs path
CREATE TABLE ecommerce_bhaviour
USING PARQUET
LOCATION '/Volumes/workspace/default/silver/ecomm-behavior';

In [0]:
df.select("user_id").distinct().count()
# 5 million users

In [0]:
df.select("user_session").distinct().count()
# 20 million sessions

In [0]:
df.createOrReplaceTempView("events")

In [0]:
%sql
CREATE TABLE ecommerce_partitioned
USING DELTA
-- USING PARQUET
-- LOCATION 'abfss://silver@<storage-account>.dfs.core.windows.net/ecommerce/partitioned/'
PARTITIONED BY (year, month)
AS
SELECT
    event_time,
    event_type,
    product_id,
    category_id,
    category_code,
    brand,
    price,
    user_id,
    user_session,
    YEAR(event_time) AS year,
    MONTH(event_time) AS month
FROM events;

In [0]:
%sql
-- bucket is not for delta table, used in hive, hive/spark, typically orc makes bucketing more efficient compared to parquet
CREATE TABLE ecommerce_bucketed
USING PARQUET
-- ORC
-- LOCATION 'abfss://silver@<storage-account>.dfs.core.windows.net/ecommerce/bucketed/'
PARTITIONED BY (year)
CLUSTERED BY (user_id)
-- CLUSTERED BY (user_id, product_id)
-- CLUSTERED BY (product_id)
INTO 32 BUCKETS
AS
SELECT
    event_time,
    event_type,
    product_id,
    category_id,
    category_code,
    brand,
    price,
    user_id,
    user_session,
    YEAR(event_time) AS year,
    MONTH(event_time) AS month
FROM events;

In [0]:
%sql
-- Liquid Clustering, we neeed to use alter table, before db runtime 14
CREATE or REPLACE TABLE ecommerce_delta_liquid_clustering
USING DELTA
-- DB Runtime 14 and above
CLUSTER BY (user_id, product_id)
AS
SELECT
    event_time,
    event_type,
    product_id,
    category_id,
    category_code,
    brand,
    price,
    user_id,
    user_session,
    YEAR(event_time) AS year,
    MONTH(event_time) AS month
FROM events;

In [0]:
%sql
ALTER TABLE ecommerce_delta_liquid_clustering
CLUSTER BY (user_id);

In [0]:
%sql
OPTIMIZE ecommerce_delta_liquid_clustering;